# Sprint 5

## Install PySpark

In [21]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [22]:
import os
import platform

if platform.system() == "Windows":
    # Point to Java 17 explicitly — required for PySpark on Windows
    os.environ["JAVA_HOME"] = r"C:\Users\bhoom\AppData\Local\Programs\Microsoft\jdk-17.0.18.8-hotspot"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
    print("Windows: Java 17 path set to", os.environ["JAVA_HOME"])
else:
    print("Non-Windows: no fix needed")

Non-Windows: no fix needed


In [23]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 4.1.1
Shuffle partitions: 8


In [24]:
from pathlib import Path
import os 

print("cwd =", Path.cwd())
print("DATA_DIR =", DATA_DIR)
print("resolved DATA_DIR =", DATA_DIR.resolve())
print("admissions path =", (DATA_DIR / "admissions.csv.gz").resolve())
print("admissions exists? =", (DATA_DIR / "admissions.csv.gz").exists())
print("patients exists? =", (DATA_DIR / "patients.csv.gz").exists())

cwd = /Users/aradhana_anandkumar/Desktop/UNDERGRAD/2nd YEAR/CS 131/bladdards-MIMIC-health
DATA_DIR = ../data/MIMIC-IV/hosp
resolved DATA_DIR = /Users/aradhana_anandkumar/Desktop/UNDERGRAD/2nd YEAR/CS 131/data/MIMIC-IV/hosp
admissions path = /Users/aradhana_anandkumar/Desktop/UNDERGRAD/2nd YEAR/CS 131/data/MIMIC-IV/hosp/admissions.csv.gz
admissions exists? = False
patients exists? = False


## Import the funtions and create data path

In [25]:
from pathlib import Path
from urllib.request import urlretrieve # to download data if not already present

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

# Path for MIMIC-IV data 
DATA_DIR = Path("data/MIMIC-IV/hosp")


## Dataframes

In [26]:
# -------------------------------------------------------
# Visits dataframe (Ara) 
# -------------------------------------------------------

# REDEFINING F 
import pyspark.sql.functions as F 
# soH (Source of Help): https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html


# 1) READING THE HOSPITAL SOURCE FILES 

# deriving (hospital-admission info) -> subject_id, adm_id, admittime, and race from admissions.csv.gz
admissions = spark.read.csv(
    str(DATA_DIR / "admissions.csv.gz"), 
    header=True, 
    inferSchema=True)

# deriving (patient-level info) -> subject_id, gender, anchor_age, anchor_year from patients.csv.gz
patients = spark.read.csv(
    str(DATA_DIR / "patients.csv.gz"), 
    header=True, 
    inferSchema=True)

# reading sprint3 output timeline
pre_bc_system_timeline = spark.read.csv(
    "../out/evidence/pre_bc_symptom_timeline.csv",
    header=True,
    inferSchema=True
)

# Extracting relevant visit-level rows from the timeline. 
# The below takes the timeline dataset, and only retrieves subject_id, hadm_id, and row_type (keep in mind, I renamed row_tyoe to visit_type to match logic) 
# It creates a list of visits of interest that can be joined into hospital data. 
visits_labeled = (
    pre_bc_symptom_timeline
    .select("subject_id", "hadm_id", "row_type")
    .withColumnRenamed("row_type", "visit_type")
)


# Here, each row represents one patient admission of interest, along with demographics and derived timing (adge + admit_day) based on the previous timeline 
# The final dataframe contains one row per (subject_id, hadm_id, visit_type); and duplicate visit entries are dropped for the same patient, admission, and visit type (as to not inflate the cleaned dataframe with duplicates)
visits = (
    visits_labeled
    .join(
        admissions.select("subject_id", "hadm_id", "admittime", "race"),
        on=["subject_id", "hadm_id"],
        how="inner"
    )
    .join(
        patients.select(
            "subject_id",
            "gender",
            "anchor_age",
            "anchor_year",
            "anchor_year_group"
        ),
        on="subject_id",
        how="inner"
    )
    .withColumn("admit_date", F.to_date("admittime"))
    .withColumn("approx_year", F.substring("anchor_year_group", 1, 4))
    .withColumn(
        "admit_day", # refined due to Sharon's commment; although, I am a little worried about implementation here becuase `anchor_year` and `approx_age` are not full dates, but just years. Not sure if Spark will make a fuss about the mismatch
        F.col("admit_date") - F.to_date("anchor_year") + F.to_date("approx_year")
    )
    .withColumn(
        "age",
        (
            F.col("anchor_age") + (F.year("admit_date") - F.col("anchor_year"))
        ).cast("int")
    )
    .select(
        F.col("subject_id").cast("int").alias("subject_id"),
        F.col("hadm_id").cast("int").alias("hadm_id"),
        F.col("race").cast("string").alias("race"),
        F.col("gender").cast("string").alias("gender"),
        F.col("visit_type").cast("string").alias("visit_type"),
        F.col("age").cast("int").alias("age"),
        F.col("admit_day")
    )
    .dropDuplicates(["subject_id", "hadm_id", "visit_type"])
)

visits.printSchema()
visits.show(20, truncate=False)



26/04/19 09:17:00 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/MIMIC-IV/hosp/admissions.csv.gz.
java.io.FileNotFoundException: File data/MIMIC-IV/hosp/admissions.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.s

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/aradhana_anandkumar/Desktop/UNDERGRAD/2nd YEAR/CS 131/bladdards-MIMIC-health/data/MIMIC-IV/hosp/admissions.csv.gz. SQLSTATE: 42K03

In [ ]:
# -------------------------------------------------------
# AGE and DATE Transformations (Ara)
# -------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# subtask 1: Frequency of Age at Diagnosis

bc_visits = visits.filter(F.col("visit_type") == "BC_FIRST_DIAGNOSIS")

bc_buckets = bc_visits.withColumn(
    "de_obfs_age_bucket",
    F.when(F.col("age") < 30, "<30")
     .when((F.col("age") >= 30) & (F.col("age") <= 40), "30-40")
     .when((F.col("age") >= 41) & (F.col("age") <= 55), "41-55")
     .when((F.col("age") >= 56) & (F.col("age") <= 70), "56-70")
     .when((F.col("age") >= 71) & (F.col("age") <= 85), "71-85")
     .otherwise("85+")
)

age_frequency = (
    bc_buckets
    .groupBy("de_obfs_age_bucket")
    .count()
    .orderBy("de_obfs_age_bucket")
)

age_frequency.show(truncate=False)


# subtask 2: Days Prior to Diagnosis

# using the timeline here because it preserves symptom/diagnosis event order
symptoms = pre_bc_system_timeline.filter(F.col("row_type") == "SYMPTOM")
bc = pre_bc_system_timeline.filter(F.col("row_type") == "BC_FIRST_DX")

symptom_counts = (
    symptoms
    .groupBy("subject_id")
    .count()
    .withColumnRenamed("count", "num_of_symptoms")
)

the_window = Window.partitionBy("subject_id").orderBy(F.col("admittime").desc())

last_symptoms = (
    symptoms
    .withColumn("rank", F.row_number().over(the_window))
    .filter(F.col("rank") == 1)
    .select("subject_id", F.col("admittime").alias("last_symptom_time"))
)

first_bc = (
    bc
    .select("subject_id", F.col("admittime").alias("bc_time"))
)

time_difference = (
    last_symptoms
    .join(first_bc, on="subject_id", how="inner")
    .join(symptom_counts, on="subject_id", how="inner")
    .withColumn(
        "days_before_dx",
        F.datediff("bc_time", "last_symptom_time")
    )
)

time_difference = time_difference.withColumn(
    "bucket",
    F.when(F.col("num_of_symptoms") == 1, "1")
     .when(F.col("num_of_symptoms") == 2, "2")
     .otherwise("3+")
)

stats_bucket = (
    time_difference
    .groupBy("bucket")
    .agg(
        F.mean("days_before_dx").alias("mean"),
        F.stddev("days_before_dx").alias("stddev"),
        F.max("days_before_dx").alias("max"),
        F.min("days_before_dx").alias("min"),
        F.expr("percentile_approx(days_before_dx, 0.5)").alias("median")
    )
    .orderBy("bucket")
)

stats_bucket.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

EVIDENCE_DIR = Path("out/evidence")
# DATA_DIR is already defined above as Path("data/MIMIC-IV/hosp")

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)


In [ ]:
# -------------------------------------------------------
# AGE and DATE Transformations (Ara)
# -------------------------------------------------------

# My section performs two analyses:
# 1) Counting how many BC diagnosis visits fall into each age range -> named Subtask 1 
# 2) Measuring how many days passed between a patient's last symptom visit
#    and their bladder cancer diagnosis visit -> named Subtask 2 

# SUBTASK 1: Frequency of Age at Diagnosis



# Keeping only visits labeled as the first breast cancer diagnosis
# Each row in 'visits' represents a visit of interest with demographic info attached
bc_visits = visits.filter(F.col("visit_type") == "BC_FIRST_DIAGNOSIS")

# Create age buckets from the derived age column
# The 'age' field was computed earlier from anchor_age and anchor_year,
# so this is a grouped version of that approximate/de-obfuscated age
bc_buckets = bc_visits.withColumn(
    "age_bucket",
    F.when(F.col("age") < 30, "<30")
     .when((F.col("age") >= 30) & (F.col("age") <= 40), "30-40")
     .when((F.col("age") >= 41) & (F.col("age") <= 55), "41-55")
     .when((F.col("age") >= 56) & (F.col("age") <= 70), "56-70")
     .when((F.col("age") >= 71) & (F.col("age") <= 85), "71-85")
     .otherwise("85+")
)

# Count how many BC diagnosis visits fall into each age bucket
age_frequency = (
    bc_buckets
    .groupBy("age_bucket")
    .count()
    .orderBy("age_bucket")
)

# Display the age distribution table
age_frequency.show(truncate=False)


# SUBTASK 2: Days Prior to Diagnosis


# Use the timeline dataframe because it preserves the ordering of
# symptom visits and diagnosis visits for each patient

# Keep only symptom rows
symptoms = pre_bc_symptom_timeline.filter(F.col("row_type") == "SYMPTOM")

# Keeping only breast cancer diagnosis rows
bc = pre_bc_symptom_timeline.filter(F.col("row_type") == "BC_FIRST_DX")

# Counting how many symptom visits each patient has in the timeline
# This will later let us compare patients with 1, 2, or 3+ symptoms
symptom_counts = (
    symptoms
    .groupBy("subject_id")
    .count()
    .withColumnRenamed("count", "num_of_symptoms")
)

# `window` definition
# Paritioning by patient and sort each patient's symptom visits by admittime descending
# Which allows us to identify the most recent symptom visit for each patient
from pyspark.sql.window import Window
the_window = Window.partitionBy("subject_id").orderBy(F.col("admittime").desc())

# For each patient, keep only the most recent symptom visit
# row_number() = 1 means "latest symptom visit" because of descending sort
last_symptoms = (
    symptoms
    .withColumn("rank", F.row_number().over(the_window))
    .filter(F.col("rank") == 1)
    .select("subject_id", F.col("admittime").alias("last_symptom_time"))
)

# Get the breast cancer diagnosis time for each patient
# Assumes one BC_FIRST_DX row per patient in this timeline
first_bc = (
    bc
    .select("subject_id", F.col("admittime").alias("bc_time"))
)

# Join:
# - the patient's last symptom time
# - the patient's diagnosis time
# - the number of symptoms that patient had
# Then compute the number of days between the last symptom visit
# and the breast cancer diagnosis visit
time_difference = (
    last_symptoms
    .join(first_bc, on="subject_id", how="inner")
    .join(symptom_counts, on="subject_id", how="inner")
    .withColumn(
        "days_before_dx",
        F.datediff("bc_time", "last_symptom_time")
    )
)

# Bucketing patients by number of symptoms:
# 1 symptom, 2 symptoms, or 3+ symptoms
time_difference = time_difference.withColumn(
    "bucket",
    F.when(F.col("num_of_symptoms") == 1, "1")
     .when(F.col("num_of_symptoms") == 2, "2")
     .otherwise("3+")
)

# For each symptom-count bucket, summary statistics are computed for days_before_dx:
# - mean
# - standard deviation
# - maximum
# - minimum
# - approximate median

# Note: the following stats were specified accordingly in the Trello Card 
stats_bucket = (
    time_difference
    .groupBy("bucket")
    .agg(
        F.mean("days_before_dx").alias("mean"),
        F.stddev("days_before_dx").alias("stddev"),
        F.max("days_before_dx").alias("max"),
        F.min("days_before_dx").alias("min"),
        F.expr("percentile_approx(days_before_dx, 0.5)").alias("median")
    )
    .orderBy("bucket")
)

# Displaying the stats for each bucket 
stats_bucket.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------


## Clean up
Stop spark session when done

In [ ]:
# Uncomment when you are completely done:

# spark.stop()